# Monte Carlo Simulation: OLS vs Newey–West HAC under AR(4) Autocorrelation

Goal: Evaluate the performance of ordinary least squares (OLS) standard errors
compared to heteroskedasticity-and-autocorrelation consistent (HAC)
Newey–West estimators when residuals follow an AR(4) process.


In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from scipy.stats import norm, chi2, t
from tqdm.notebook import trange, tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats

In [44]:
# Simulation parameters
np.random.seed(2025)
R = 1000                # number of replications
T = 100                 # sample size
betas_true = np.array([0.0, 1.0, 0.5, -0.5])
phi = np.array([0.4, -0.2, 0.15, -0.05])  # AR(4) coefficients
bandwidth = int(4 * (T/100)**(2/9))  # HAC lag lengths


In [45]:
def simulate_ar4(T, phi, sigma=1.0):
    epsilon = np.random.normal(0, sigma, T)
    u = np.zeros(T)
    for t in range(4, T):
        u[t] = phi[0]*u[t-1] + phi[1]*u[t-2] + phi[2]*u[t-3] + phi[3]*u[t-4] + epsilon[t]
    return u

def one_replication(T, phi, betas, bandwidth):
    """
    Run one Monte Carlo replication comparing OLS and Newey-West estimators.

    Returns a dictionary with:
    betahat, sandwich variance, beta variance, confidence intervals, and p-values.
    """

    # --- 1️⃣ Simulate regressors ---
    X = np.column_stack([
        np.ones(T),
        np.random.normal(size=T),
        np.random.normal(size=T),
        np.random.normal(size=T)
    ])
    # --- 2️⃣ Simulate AR(4) errors ---
    u = simulate_ar4(T,phi)

    # --- 3️⃣ Generate dependent variable ---
    y = np.matmul(X,betas) + u

    # --- 4️⃣ Fit OLS model ---
    model = sm.OLS(y, X).fit()
    betahat = np.linalg.inv(X.T @ X) @ X.T @ y
    k = len(betahat)
    df = T - k

    # --- 5️⃣ Initialize results dictionary ---
    results = {}

    # --- 6️⃣ Compute OLS quantities ---

    residuals = (y - X @ betahat)
    SSR = np.dot(residuals,residuals)
    SST = np.dot(y - np.mean(y), y -np.mean(y))
    R2_ols = 1 - SSR / SST
    R2_adj_ols = 1 - (SSR / df) / (SST / (T-1))
    s2_ols = SSR / df
    cov_beta_ols = s2_ols * np.linalg.inv(X.T @ X)
    se_beta_ols = np.sqrt(np.diag(cov_beta_ols))
    t_stats_ols = betahat / se_beta_ols
    p_value_ols = 2 * (1 - t.cdf(np.abs(t_stats_ols),df))
    z_95 = norm.ppf(1-0.05/2)
    ci_asymp_ols_95 = np.column_stack([betahat - z_95*se_beta_ols, betahat + z_95*se_beta_ols])
    residuals_studentized = residuals / (np.sqrt(s2_ols) * np.sqrt(1 - np.diag(X @ np.linalg.inv(X.T @ X) @ X.T))) 

    results['OLS'] = {
        'betahat': betahat,
        'residuals': residuals,
        'residuals_studentized': residuals_studentized,
        's2': s2_ols,
        'R2': R2_ols,
        'R2_adj': R2_adj_ols,
        'cov_beta': cov_beta_ols,
        'se_beta': se_beta_ols,
        't_stats': t_stats_ols,
        'p_value': p_value_ols,
        'conf_int_95': ci_asymp_ols_95,
    }

    # --- 7️⃣ Compute Newey–West quantities ---
    cov_nw = cov_hac(model, nlags=bandwidth)
    se_nw = np.sqrt(np.diag(cov_nw))
    t_nw = betahat / se_nw
    p_nw = 2 * (1 - norm.cdf(np.abs(t_nw)))
    ci_nw_95 = np.column_stack([betahat - z_95*se_nw, betahat + z_95*se_nw])

    results[f'NW'] = {
        'lag': bandwidth,
        'var_sandwich': cov_nw,
        'cov_beta': np.diag(cov_nw),
        'conf_int_95': ci_nw_95,
        'p_value': p_nw
    }

    return results



In [46]:
results = one_replication(T, phi, betas_true, bandwidth)
for key, value in results['OLS'].items():
    if key != 'residuals' and key != 'residuals_studentized':
        print(f"{key}: {value}")

betahat: [-0.266 0.95 0.371 -0.569]
s2: 0.9554004516546991
R2: 0.53953164855682
R2_adj: 0.5251420125742208
cov_beta: [[0.00985 0.00158 -0.000552 0.000415]
 [0.00158 0.0114 0.00155 -0.000359]
 [-0.000552 0.00155 0.0104 -0.000148]
 [0.000415 -0.000359 -0.000148 0.0105]]
se_beta: [0.0992 0.107 0.102 0.102]
t_stats: [-2.68 8.9 3.63 -5.56]
p_value: [0.0087 3.46e-14 0.000452 2.37e-07]
conf_int_95: [[-0.46 -0.0713]
 [0.74 1.16]
 [0.171 0.57]
 [-0.77 -0.369]]


In [47]:
np.set_printoptions(formatter={'float_kind': '{:0.3}'.format})
results["OLS"]["conf_int_95"]

array([[-0.46, -0.0713],
       [0.74, 1.16],
       [0.171, 0.57],
       [-0.77, -0.369]])

In [48]:
print(results[f"NW"]["conf_int_95"])

[[-0.515 -0.0168]
 [0.791 1.11]
 [0.141 0.6]
 [-0.744 -0.395]]


In [49]:
np.set_printoptions(formatter={'float_kind': '{:0.3e}'.format})
results["OLS"]["p_value"]

array([8.695e-03, 3.464e-14, 4.520e-04, 2.372e-07])

In [50]:
print(results[f"NW"]["p_value"])

[3.639e-02 0.000e+00 1.523e-03 1.710e-10]


In [51]:
residuals = results['OLS']['residuals']
residuals_studentized = results['OLS']['residuals_studentized']

# --- QQ plot (Plotly) ---
qq_theoretical = np.sort(stats.norm.rvs(size=T))
qq_empirical = np.sort(residuals_studentized)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=qq_theoretical,
    y=qq_empirical,
    mode='markers',
    name='Studentized residuals',
    marker=dict(color='rgba(50,100,200,0.6)')
))

# 45° reference line
min_val = min(qq_theoretical.min(), qq_empirical.min())
max_val = max(qq_theoretical.max(), qq_empirical.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Normal line',
    line=dict(color='red', dash='dash')
))

fig.update_layout(
    title=dict(text="QQ Plot of Studentized Residuals", x=0.5, xanchor='center'),
    xaxis_title="Theoretical Quantiles (Normal)",
    yaxis_title="Empirical Quantiles (Studentized Residuals)",
    width=800,
    height=600,
    template='plotly_white',
    legend=dict(
        font=dict(size=14),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5
    )
)

fig.show()

In [57]:
# --- Normality tests for different values of sample sizes T ---
sample_sizes = [100, 500, 1000]

for T in sample_sizes:
    results = one_replication(T, phi, betas_true, bandwidth)
    residuals = results['OLS']['residuals']
    residuals_studentized = results['OLS']['residuals_studentized']

    # --- Normality tests ---
    # Jarque–Bera test
    jb_stat, jb_p = stats.jarque_bera(residuals_studentized)

    # Shapiro–Wilk test
    sw_stat, sw_p = stats.shapiro(residuals_studentized)

    # Kolmogorov–Smirnov test
    ks_stat, ks_p = stats.kstest(residuals_studentized, 'norm')

    # --- Collect results ---
    tests_results = {
        'Jarque–Bera': {
            'statistic': jb_stat,
            'p_value': jb_p,
            'reject_null': jb_p < 0.05,
            'sample_size': T
        },
        'Shapiro–Wilk': {
            'statistic': sw_stat,
            'p_value': sw_p,
            'reject_null': sw_p < 0.05,
            'sample_size': T
        },
        'Kolmogorov–Smirnov': {
            'statistic': ks_stat,
            'p_value': ks_p,
            'reject_null': ks_p < 0.05,
            'sample_size': T
        }
    }
    display(pd.DataFrame(tests_results).T)


,statistic,p_value,reject_null,sample_size
Jarque–Bera,1.001245,0.606153,False,100
Shapiro–Wilk,0.992616,0.86383,False,100
Kolmogorov–Smirnov,0.045208,0.981101,False,100


,statistic,p_value,reject_null,sample_size
Jarque–Bera,2.070429,0.35515,False,500
Shapiro–Wilk,0.996785,0.425254,False,500
Kolmogorov–Smirnov,0.031045,0.708777,False,500


,statistic,p_value,reject_null,sample_size
Jarque–Bera,2.566251,0.27717,False,1000
Shapiro–Wilk,0.998368,0.471226,False,1000
Kolmogorov–Smirnov,0.022398,0.6887,False,1000


In [53]:
def montecarlo_analysis(T, phi, betas, bandwidth, n_rep=10000):
    """Run Monte Carlo to check CI coverage and p-values"""
    p = len(betas)
    betahat_distr = np.zeros((n_rep,p))

    for i in trange(n_rep):
        res = one_replication(T, phi, betas, bandwidth)
        betahat_distr[i,:] = res['OLS']['betahat']
    return np.array(betahat_distr)

# Example usage
T = 200
bandwidth = int(4 * (T/100)**(2/9))

betahat_distr = montecarlo_analysis(T, phi, betas_true, bandwidth)
np.set_printoptions(formatter={'float_kind': '{:0.3}'.format})
print(betahat_distr)

  0%|          | 0/10000 [00:00<?, ?it/s]

[[-0.14 0.911 0.453 -0.331]
 [-0.0015 1.03 0.548 -0.59]
 [-0.108 0.961 0.494 -0.536]
 ...
 [0.0199 0.973 0.496 -0.476]
 [-0.0833 0.97 0.452 -0.472]
 [-0.0486 1.03 0.373 -0.492]]


In [54]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

k = betahat_distr.shape[1]
n_rows, n_cols = 2, 2

# --- Compute global y-axis limit for consistent density scale ---
all_counts = []
for i in range(k):
    hist, _ = np.histogram(betahat_distr[:, i], bins=100, density=True)
    all_counts.append(np.max(hist))
y_max = max(all_counts) * 1.1

# --- Create subplots ---
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"β{i}" for i in range(k)],
    horizontal_spacing=0.12,
    vertical_spacing=0.15
)

# --- Add each β̂ histogram + vertical lines ---
for i in range(k):
    row = i // n_cols + 1
    col = i % n_cols + 1
    beta_true = betas_true[i]
    beta_mean = np.mean(betahat_distr[:, i])

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=betahat_distr[:, i],
            nbinsx=100,
            histnorm='probability density',
            marker_color='rgba(0, 90, 180, 0.6)',
            showlegend=False
        ),
        row=row, col=col
    )

    # Vertical lines: true β (green) and sample mean (red dashed)
    fig.add_vline(
        x=beta_true,
        line=dict(color='green', width=3),
        row=row, col=col
    )
    fig.add_vline(
        x=beta_mean,
        line=dict(color='red', width=2, dash='dash'),
        row=row, col=col
    )

# --- Add dummy traces for legend ---
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                         line=dict(color='green', width=3),
                         name='True β'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                         line=dict(color='red', width=2, dash='dash'),
                         name='Sample Mean β_OLS'))

# --- Style adjustments ---
fig.update_xaxes(title_text="β_OLS value", title_font=dict(size=14))
fig.update_yaxes(range=[0, y_max], title_text="Density", title_font=dict(size=14))

fig.update_layout(
    height=900,
    width=1150,
    title={
        'text': "Monte Carlo Distributions of β_OLS Estimates",
        'x': 0.5,  # center title
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=22, family="Arial Bold")
    },
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.18,
        xanchor="center",
        x=0.5,
        font=dict(size=15, family="Arial", color="black"),
        bgcolor="rgba(255,255,255,0.6)",
        bordercolor="lightgray",
        borderwidth=1
    ),
    margin=dict(l=70, r=70, t=100, b=100)
)

fig.show()


In [55]:
# --- Monte Carlo test of empirical size at 1%, 5%, and 10% ---
print(results['NW']['p_value'])

n_rep = 10000
alpha_levels = [0.01, 0.05, 0.10]

# Initialize counters: one row per alpha, one column per beta
counter_ols = np.zeros((len(alpha_levels), len(betas_true)))
counter_nw  = np.zeros((len(alpha_levels), len(betas_true)))

for _ in trange(n_rep, desc="Monte Carlo replications"):
    results = one_replication(T, phi, betas_true, bandwidth)
    p_value_ols = np.array(results['OLS']['p_value'])
    p_value_nw  = np.array(results['NW']['p_value'])

    for j, alpha in enumerate(alpha_levels):
        counter_ols[j] += (p_value_ols < alpha).astype(int)
        counter_nw[j]  += (p_value_nw < alpha).astype(int)

# Compute empirical rejection frequencies
reject_rate_ols = counter_ols / n_rep
reject_rate_nw  = counter_nw / n_rep


[0.836 0.0 0.0 0.0]


Monte Carlo replications:   0%|          | 0/10000 [00:00<?, ?it/s]

In [56]:
# turn into DataFrames for better display
df_ols = pd.DataFrame(reject_rate_ols, index=[f'α={a}' for a in alpha_levels],
                      columns=[f'β{i}' for i in range(len(betas_true))])
df_nw  = pd.DataFrame(reject_rate_nw, index=[f'α={a}' for a in alpha_levels],
                      columns=[f'β{i}' for i in range(len(betas_true))])
print("Empirical rejection rates (OLS):")
display(df_ols)
print("Empirical rejection rates (Newey–West):")
display(df_nw)

Empirical rejection rates (OLS):


,β0,β1,β2,β3
α=0.01,0.0522,1.0,0.9999,1.0
α=0.05,0.1362,1.0,1.0000,1.0
α=0.1,0.2146,1.0,1.0000,1.0


Empirical rejection rates (Newey–West):


,β0,β1,β2,β3
α=0.01,0.0223,1.0,0.9997,0.9998
α=0.05,0.0737,1.0,1.0000,1.0000
α=0.1,0.1308,1.0,1.0000,1.0000
